# Testiranje `agg_posek_meritve.py` (v2)
Preverja `agg_temp`, `agg_precip` in `_best_window_df` z resničnimi podatki.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

_search = Path(os.getcwd()).resolve()
ROOT = None
for _ in range(5):
    if (_search / "src").exists() and (_search / "data").exists():
        ROOT = _search
        break
    _search = _search.parent
if ROOT is None:
    ROOT = Path("/home/anejm/Documents/hekaton/BarkWatch-arnes_hackathon2026")

sys.path.insert(0, str(ROOT / "src" / "data_processing"))

from agg_posek_meritve import agg_temp, agg_precip, _best_window_df, WINDOWS
from bliznje_vremenske_postaje import get_postaje, _load_vreme

print(f"ROOT: {ROOT}")
print("Uvoz uspešen.")

## 1. Naloži vreme in posek

In [ ]:
vreme_all = _load_vreme()
available_stations = set(vreme_all.index.get_level_values("station_id"))
print(f"Postaje v vreme.csv: {len(available_stations)}")
print(f"Obdobje: {vreme_all.index.get_level_values('datum').min().date()} – "
      f"{vreme_all.index.get_level_values('datum').max().date()}")

In [ ]:
df_posek = pd.read_csv(ROOT / "data" / "raw" / "ZGS" / "posek.csv", low_memory=False)
df_posek.columns = df_posek.columns.str.strip()
df_posek["odsek"] = df_posek["odsek"].str.strip()
df_posek["posekano"] = pd.to_datetime(df_posek["posekano"], errors="coerce")
df_posek = df_posek.dropna(subset=["odsek", "posekano"]).reset_index(drop=True)
print(f"Posek vrstice: {len(df_posek):,}  |  Odseki: {df_posek['odsek'].nunique()}")

## 2. Test na enem odseku — ročni pregled

In [ ]:
# Vzemi prvi odsek s posekom po letu 2010
vzorec_row = df_posek[df_posek["posekano"].dt.year >= 2010].iloc[0]
ODSEK    = vzorec_row["odsek"]
END_DATE = vzorec_row["posekano"]
print(f"Odsek: {ODSEK}  |  Datum poseka: {END_DATE.date()}")

postaje = get_postaje(ODSEK)
sids_temp = [str(p.id) for p in postaje["temp"]]
sids_all  = [str(p.id) for p in postaje["all"]]
print(f"\nTop-3 temp postaje: {postaje['temp']}")
print(f"Top-3 all  postaje: {postaje['all']}")

In [ ]:
# Preveri katera postaja ima podatke za vsako okno
for win_name, days in WINDOWS.items():
    df_t = _best_window_df(sids_temp, vreme_all, available_stations, END_DATE, days, "povp_T_degC")
    df_p = _best_window_df(sids_all,  vreme_all, available_stations, END_DATE, days, "padavine_mm")
    src_t = f"{len(df_t)} dni" if df_t is not None else "NI PODATKOV"
    src_p = f"{len(df_p)} dni" if df_p is not None else "NI PODATKOV"
    print(f"{win_name:5s}  temp: {src_t:12s}  padavine: {src_p}")

In [ ]:
# Agregiraj in prikaži vse vrednosti za 30d okno
df_t = _best_window_df(sids_temp, vreme_all, available_stations, END_DATE, 30, "povp_T_degC")
df_p = _best_window_df(sids_all,  vreme_all, available_stations, END_DATE, 30, "padavine_mm")

rt = agg_temp(df_t, "30d")
rp = agg_precip(df_p, "30d")

print("Temperatura (30d):")
for k, v in rt.items(): print(f"  {k}: {v}")
print("\nPadavine (30d):")
for k, v in rp.items(): print(f"  {k}: {v}")

## 3. Vzorec 20 parov — pokritost

In [ ]:
N = 20
vzorec = (
    df_posek[df_posek["posekano"].dt.year >= 2010]
    [["odsek", "posekano"]].drop_duplicates()
    .sample(N, random_state=42).reset_index(drop=True)
)

rows = []
for _, row in vzorec.iterrows():
    odsek, end_date = row["odsek"], row["posekano"]
    try:
        p = get_postaje(odsek)
        sids_t = [str(x.id) for x in p["temp"]]
        sids_a = [str(x.id) for x in p["all"]]
    except Exception:
        sids_t, sids_a = [], []

    feat = {"odsek": odsek, "posekano": end_date}
    for win_name, days in WINDOWS.items():
        df_t = _best_window_df(sids_t, vreme_all, available_stations, end_date, days, "povp_T_degC")
        df_p = _best_window_df(sids_a, vreme_all, available_stations, end_date, days, "padavine_mm")
        feat.update(agg_temp(df_t, win_name))
        feat.update(agg_precip(df_p, win_name))
    rows.append(feat)

df_rez = pd.DataFrame(rows)

# Pokritost po stolpcih
vreme_cols = [c for c in df_rez.columns if c not in ("odsek", "posekano")]
pokritost = df_rez[vreme_cols].notna().mean().mul(100).round(1)
print(f"Pokritost ({N} parov):")
print(pokritost.to_string())

In [ ]:
df_rez